# E07：数组筛选、聚合、缺失值与广播

**课程**：Python 程序设计  
**适用对象**：应用统计学专业一年级  
**实验类型**：验证性实验  
**建议学时**：2 学时

## 实验目标

完成本实验后，你应当能够：

1. 使用 `np.nan`、`np.isnan()` 和布尔掩码处理数值缺失。
2. 组合多个布尔条件完成筛选和分组汇总。
3. 使用二维数组和 `axis` 明确按行与按列的处理方向。
4. 直观使用广播完成按列阈值判断，并核验筛选前后的统计口径。

## 实验信息

- 姓名：  
- 学号：  
- 班级：  
- 日期：

## 提交要求

1. 在本 Notebook 中完成全部任务，不删除题目、提示和课程提供的数据单元。
2. 保留最终可运行的代码、输出结果和必要的 Markdown 说明。
3. 不修改或覆盖 `lab/shared` 中的原始数据和官方检查点。
4. 提交前重启内核并按顺序运行全部单元，检查数组形状、样本量和统计口径。
5. 如使用帮助文档、代码补全或 AI 工具，应在实验小结中说明用途和核验方式。

## Jupyter 使用提示

- 先运行输入准备单元，再完成后续任务。
- 修改代码后必须重新运行当前单元，输出才会更新。
- 数组计算结果异常时，先检查 `shape`、缺失值和布尔掩码。
- 如果变量状态混乱，重启内核并从上到下重新运行。

## 建议节奏

- 保留缺失值并创建对齐数组：15 分钟
- 缺失值识别与有效值聚合：15 分钟
- 布尔筛选与组合条件：15 分钟
- 二维数组和 `axis`：15 分钟
- 广播与综合汇总：15 分钟
- 调试、实验小结与提交：15 分钟

## 主线延续：保持 160 条记录对齐

本实验继续使用与 E06 相同的输入选择和核验规则。与 E06 删除缺失记录不同，E07 使用 `np.nan` 保留原有 160 个位置，使数值数组与记录列表保持对齐。字符串字段不组成混合数组，而是直接转换为同长度的布尔掩码。

In [ ]:
import json
from pathlib import Path

import numpy as np

# 有可靠的 E05 个人成果时保持 True；需要从官方检查点重新开始时改为 False。
USE_STUDENT_E05_OUTPUT = True

course_lab_dir = Path("..")
if not (course_lab_dir / "chapter2").exists():
    course_lab_dir = Path("lab/applied-stat-64")

shared_dir = Path("../../shared")
if not shared_dir.exists():
    shared_dir = Path("lab/shared")

student_path = (
    course_lab_dir
    / "chapter2"
    / "e05_output"
    / "e05_cleaned_records.json"
)
official_path = (
    shared_dir
    / "checkpoints"
    / "e05_cleaned_records.json"
)

if USE_STUDENT_E05_OUTPUT and student_path.exists():
    input_path = student_path
    input_source = "学生 E05 输出"
else:
    input_path = official_path
    input_source = "官方 E05 检查点"
    if USE_STUDENT_E05_OUTPUT:
        print("未找到学生 E05 输出，改用官方检查点。")

with open(input_path, "r", encoding="utf-8") as file:
    records = json.load(file)

expected_fields = [
    "student_id",
    "alias",
    "gender",
    "major",
    "prior_programming",
    "support_plan",
    "reported_study_hours",
    "reported_attendance_rate",
]
student_ids = [record.get("student_id") for record in records]
missing_hours = sum(
    record.get("reported_study_hours") is None
    for record in records
)
missing_attendance = sum(
    record.get("reported_attendance_rate") is None
    for record in records
)

input_checks = {
    "记录数为 160": len(records) == 160,
    "student_id 唯一": len(set(student_ids)) == len(student_ids),
    "字段及顺序一致": all(
        list(record.keys()) == expected_fields
        for record in records
    ),
    "学习时长缺失数为 8": missing_hours == 8,
    "出勤率缺失数为 4": missing_attendance == 4,
}

print("输入来源：", input_source)
print("输入文件：", input_path.resolve())
for check_name, passed in input_checks.items():
    print(check_name, "：", passed)

if not all(input_checks.values()):
    raise ValueError(
        "输入核验未通过：请检查 E05 输出，"
        "或把 USE_STUDENT_E05_OUTPUT 改为 False。"
    )

## 任务 1：使用 `np.nan` 保留缺失位置

建立与 `records` 等长的学习时长和出勤率浮点数组。原字段为 `None` 时写入 `np.nan`；同时建立 `feedback_mask`，标记支持方案为 `feedback` 的记录。

In [ ]:
# TODO：把 None 转换为 np.nan，建立两个等长浮点数组
study_hours = np.full(len(records), np.nan)
attendance_rates = np.full(len(records), np.nan)

# TODO：建立 support_plan 为 feedback 的布尔掩码
feedback_mask = np.zeros(len(records), dtype=bool)

print("三个数组 shape：", study_hours.shape, attendance_rates.shape, feedback_mask.shape)

## 任务 2：识别缺失值并计算有效统计量

分别建立两个缺失掩码，统计缺失数量。只对非缺失学习时长计算有效样本量、均值、最小值、最大值和标准差。

In [ ]:
missing_hours_mask = np.isnan(study_hours)
missing_attendance_mask = np.isnan(attendance_rates)

# TODO：建立非缺失掩码并筛选有效学习时长
valid_hours_mask = np.zeros(len(records), dtype=bool)
valid_hours = np.array([], dtype=float)

print("学习时长缺失数：", missing_hours_mask.sum())
print("出勤率缺失数：", missing_attendance_mask.sum())
# TODO：输出有效样本量、均值、最小值、最大值和标准差

## 任务 3：组合布尔掩码完成分组统计

组合“学习时长非缺失”和支持方案掩码，分别计算 `feedback` 与 `regular` 方案的有效样本量和平均学习时长。

In [ ]:
# TODO：组合非缺失掩码与支持方案掩码
feedback_valid_mask = valid_hours_mask & feedback_mask
regular_valid_mask = valid_hours_mask & ~feedback_mask

# TODO：筛选两组学习时长并输出样本量和均值

## 任务 4：用二维数组检查完整记录

按“学习时长、出勤率”两列建立 `numeric_matrix`。使用 `axis=1` 找出任一字段缺失的行，筛选得到 `complete_matrix`；再使用 `axis=0` 计算两列均值。

In [ ]:
# TODO：按“学习时长、出勤率”两列建立二维数组
numeric_matrix = np.column_stack((study_hours, attendance_rates))

# TODO：使用 np.isnan(...).any(axis=1) 找出不完整行
incomplete_row_mask = np.zeros(len(records), dtype=bool)
complete_matrix = numeric_matrix[~incomplete_row_mask]

# TODO：使用 axis=0 计算完整记录的两列均值

## 任务 5：直观使用广播完成按列判断

两列的参考阈值分别为每周学习 8 小时、出勤率 80%。使用一维阈值数组与二维完整记录直接比较，检查广播后的形状，并统计同时达到两个阈值的记录数。

In [ ]:
thresholds = np.array([8.0, 80.0])

# TODO：利用广播逐列比较完整记录和阈值
meets_thresholds = np.zeros(complete_matrix.shape, dtype=bool)

# TODO：使用 all(axis=1) 判断每行是否同时达到两个阈值
meets_both = np.zeros(complete_matrix.shape[0], dtype=bool)

print("完整记录 shape：", complete_matrix.shape)
print("阈值 shape：", thresholds.shape)
print("比较结果 shape：", meets_thresholds.shape)

## 任务 6：完成综合口径核验

核验缺失数、完整记录数和分组有效样本量。说明为什么不能把学习时长与出勤率直接按行求平均，并输出一个包含关键结果的摘要字典。

In [ ]:
summary = {
    "record_count": len(records),
    # TODO：补全缺失数、完整记录数、两组有效样本量和双阈值人数
}

print(summary)
# TODO：核验两组有效样本量之和是否等于全部有效学习时长数

## 实验小结

请用 Markdown 回答：

1. 为什么使用 `np.nan` 后仍要同时报告有效样本量？
2. `axis=0` 和 `axis=1` 在本实验中分别沿什么方向处理？
3. 为什么不能把学习时长和出勤率直接按行求平均？

### 实验小结

- 